<center><h1>Un-rooted N-array Trees Isomorphism</center></h1>
The aim of this notebook is to show how to check for isomorphism for any connected nodes (un-rooted n-array trees). The explanation is not detailed. However, we will explain step by step as we go through the process
This is based on the problem from HackerRank called Jenny's Subtrees. I recommend you to check out the problem before continuing any further to understand the approach even better.
Another approach is to hold all the leaves of any given tree and start hashing the tree from the leaves. Because for any isomorphic tree, no matter what root you choose, the leaves will always be the same. Therefore, this is also a very solid approach.

### The following cell produces functions to help with visualization which will be helpful if you need some visualization

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from ipywidgets import VBox, Output, Layout

def hierarchy_pos(graph, root=None, width=1.0, vert_gap=0.2, vert_loc=0, x_center=0.5, pos=None, parent=None):
    """
    A custom hierarchical layout for a tree graph.
    """
    if pos is None:
        pos = {}
    if root is None:
        # Choose arbitrary root if none is provided
        root = list(graph.nodes)[0]

    pos[root] = (x_center, vert_loc)
    children = list(graph.neighbors(root))
    if not isinstance(graph, nx.DiGraph) and parent is not None:
        children.remove(parent)  # Remove the parent in an undirected graph

    if len(children) != 0:
        dx = width / len(children)
        nextx = x_center - width / 2 - dx / 2
        for child in children:
            nextx += dx
            pos = hierarchy_pos(
                graph, root=child, width=dx, vert_gap=vert_gap,
                vert_loc=vert_loc - vert_gap, x_center=nextx, pos=pos, parent=root
            )
    return pos

def plot_tree(root):
    # Create a directed graph
    graph = nx.DiGraph()
    node_labels = {}  # Dictionary to store node values for labeling

    # Use a stack for iterative traversal and a set to track visited nodes
    stack = [(root, None)]  # (current_node, parent_node)
    visited = set()

    while stack:
        current, parent = stack.pop()
        current_id = id(current)  # Use unique ID for each node

        if current_id in visited:
            continue
        visited.add(current_id)

        # Add node label (node value)
        node_labels[current_id] = current.value

        # Add an edge if there is a parent
        if parent is not None:
            graph.add_edge(id(parent), current_id)

        # Add children to the stack for traversal
        for child in current.children:
            if id(child) not in visited:
                stack.append((child, current))

    # Generate positions for the nodes using the custom hierarchy layout
    pos = hierarchy_pos(graph, root=id(root))

    # Plot the graph
    plt.figure(figsize=(15, 10))
    nx.draw(graph, pos, with_labels=False, arrows=False, node_size=500, node_color="lightblue")
    nx.draw_networkx_labels(graph, pos, labels=node_labels, font_size=15, font_color="black")
    plt.show()



def plot_tree_in_widget(root):
    """
    Creates a plot for a single tree and returns it as an ipywidgets Output widget.
    """
    out = Output()
    with out:
        # Create a directed graph
        graph = nx.DiGraph()
        node_labels = {}  # Dictionary to store node values for labeling

        # Use a stack for iterative traversal and a set to track visited nodes
        stack = [(root, None)]  # (current_node, parent_node)
        visited = set()

        while stack:
            current, parent = stack.pop()
            current_id = id(current)  # Use unique ID for each node

            if current_id in visited:
                continue
            visited.add(current_id)

            # Add node label (node value)
            node_labels[current_id] = current.value

            # Add an edge if there is a parent
            if parent is not None:
                graph.add_edge(id(parent), current_id)

            # Add children to the stack for traversal
            for child in current.children:
                if id(child) not in visited:
                    stack.append((child, current))

        # Generate positions for the nodes using the custom hierarchy layout
        pos = hierarchy_pos(graph, root=id(root))

        # Plot the graph
        plt.figure(figsize=(15, 10))
        nx.draw(graph, pos, with_labels=False, arrows=False, node_size=500, node_color="lightblue")
        nx.draw_networkx_labels(graph, pos, labels=node_labels, font_size=15, font_color="black")
        plt.title("Tree Visualization")
        plt.show()
    return out

def plot_multiple_trees_in_scrollable_widget(roots):
    """
    Plots multiple tree graphs in a scrollable widget container.
    """
    outputs = [plot_tree_in_widget(root) for root in roots]
    container = VBox(outputs, layout=Layout(max_height="800px", overflow="auto", border="1px solid black"))
    return container

### Let's create our simple Node class.
For further optimization more data can be stored during creation to be used for easier tree dissection. I was also thinking of a potential N-Dimensions numpy array that can present the tree in N-dimensions. Assuming this approach is possible, it will make dissection and finding the center of any portion way easier. However, I don't have the time to invest into that currently

In [ ]:
class Node:
    def __init__(self, value, children=None):
        self.value = value
        if children is None:
            children = []
        self.children = children

### Let's create a hasher function. 
The hasher function job is to create a unique hash for any unique tree structure. We will ignore children arrangement by sorting their hash on each node.

In [ ]:
import hashlib

def hasher(node, visited):
    # Base case
    if not node.children :
        return hashlib.blake2b(b'None').hexdigest()
    
    # Store kids hash in Kids then sort them to ignore arrangements
    kids=[]
    visited.add(node)
    for ele in node.children:
        if ele in visited: continue
        else:
            visited.add(ele)
            kids.append(hasher(ele, visited))
    kiddo= sorted(kids)
    final_hash =hashlib.blake2b(str(kiddo).encode()).hexdigest()
    return final_hash

### Let's create a cutter function that cuts a tree around any given node.
It returns the root of the new tree (We use a new tree for the cutted version). It's also possible to completely skip this function by using a radius in the hasher function and the center function which will paypass this step completely. However, it's effect on the overall complexity isn't much because we still need to find the center of the new tree. This function takes one step into that by returning the furthest point from the root.

In [ ]:
def cutter(node, r, root=None, visited=None, vault=None):
    # Base case if radius is at boundry
    if r <=0: return None

    # Creation of a new root to the new tree copy
    if root== None: 
        root=Node(node.value)
    if visited == None: visited = set()

    #Creation of a vault to store depths. This is originally made to be used for one-shot center allocation, but I stopped chasing after it.
    if vault == None: 
        vault= {}
        vault[root] = 0
        vault['max']= (root,0)

    visited.add(node)
    
    # Loop recursevly to generate the new tree
    for ele in node.children:
        if ele in visited: continue
        else:
            visited.add(ele)
            nono=Node(ele.value)
            root.children.append(nono)
            nono.children.append(root)
            vault[nono]=vault[root]+1
            if vault[nono]>vault['max'][1]: vault['max']=(nono,vault[nono])
            cutter(ele, r-1, nono, visited, vault)
    return root, vault

### Let's create a function that finds the center.
Since we found the furthest point from the root in the cutter function, let's call it x1. We will locate the furthest point from x1, calling it x2. Then the center node or two nodes will be at the middle of the path between x1 and x2.

In [1]:
from collections import deque
def find_center_from_x1(x1):
    # Step 1: Perform BFS from x1 to find the furthest node x2 and track parents
    def bfs(start):
        queue = deque([start])  # Queue for BFS
        visited = set([start])  # To avoid revisiting nodes
        parent_map = {start: None}  # Track the parent of each node
        distance_map = {start: 0}  # Track the distance from the start node
        max_distance = 0
        farthest_node = start
        
        while queue:
            node = queue.popleft()
            
            # Traverse children (ignoring the parent to avoid cycles)
            for child in node.children:
                if child not in visited:
                    visited.add(child)
                    queue.append(child)
                    parent_map[child] = node  # Set parent for the child
                    distance_map[child] = distance_map[node] + 1  # Update distance from start
                    
                    # Update the furthest node and max distance
                    if distance_map[child] > max_distance:
                        max_distance = distance_map[child]
                        farthest_node = child
        
        return farthest_node, parent_map, distance_map

    # Step 2: Find the furthest node (x2) from x1
    x2, parent_map, distance_map = bfs(x1)

    # Step 3: Reconstruct the path from x1 to x2 using parent_map
    def reconstruct_path(x2, parent_map):
        path = []
        node = x2
        while node is not None:
            path.append(node)
            node = parent_map[node]
        path.reverse()  # Reverse to get path from x1 to x2
        return path

    # Get the path from x1 to x2
    path = reconstruct_path(x2, parent_map)

    # Step 4: Return the center(s) of the path
    n = len(path)
    if n % 2 == 0:
        return [path[n // 2 - 1], path[n // 2]]  # Two centers if even length
    else:
        return [path[n // 2]]  # Single center if odd length


### Let's connect all the dots on a main function
This function will create all nodes from the given data, cut the tree on each node based on radius, find the centers, hash all these trees, save them to a set, and Finally count the unique trees.

In [2]:
def jennysSubtrees(n, r, edges):
    # Create Nodes from 1 to n
    nodes = [Node(i+1) for i in range(n)]

    # Build the tree from the edges
    for ele in edges:
        nodes[ele[0] - 1].children.append(nodes[ele[1] - 1])
        nodes[ele[1] - 1].children.append(nodes[ele[0] - 1])

    # Storage variables
    trees=set()
    counter=0
    
    # For each node in the tree, we will cut, center, hash, then compare.
    for node in nodes:
        # Cut and return the root and furthest point from the root x1
        new, vault = cutter(node, r)

        # Using x1 from the cutter function we calculate the center
        centers= find_center_from_x1(vault['max'][0])

        # Tracking whether the current tree is found or not. Pair is for the center pairs
        found = False
        pair = None

        for ele in centers:
            # Hashing the tree from the current center
            hashing = hasher(ele, set())
            if hashing == pair: break           # If both trees from the centr are the same
            pair = hashing

            if hashing in trees: 
                found = True 
                break
            else:
                trees.add(hashing)
        if not found: 
            counter+=1

    return  counter

### Driver code to read form input.txt and execute the algorithm

In [ ]:
inp = open('input.txt', 'r')
first_multiple_input = inp.readline().rstrip().split()
n = int(first_multiple_input[0])
r = int(first_multiple_input[1])
edges = []

for _ in range(n - 1):
    edges.append(list(map(int, inp.readline().rstrip().split())))

counter = jennysSubtrees(n, r, edges)
print( counter)

### A Final Note:
You can use the visualization tools provided in the first Python cell to plot and visualize any tree from any node, or visualize multiple trees using a list of nodes, each node corresponds to a root. This may come in handy if you are looking to understand how the trees change shape based on the selected root and how this affects the hashing value.

This is but a simple helper tool. It can be developed for further visualizations to make it easier to understand the task, but no time to invest.

In [ ]:
# Let's assume a simple example from the input data

# Create Nodes from 1 to n
nodes = [Node(i+1) for i in range(n)]

# Build the tree from the edges
for ele in edges:
    nodes[ele[0] - 1].children.append(nodes[ele[1] - 1])
    nodes[ele[1] - 1].children.append(nodes[ele[0] - 1])

# Visualizng a single tree
plot_tree_in_widget(nodes[0])

# Visualizing multiple trees
plot_multiple_trees_in_scrollable_widget(nodes)